# CLA 4 — ML Model Deployment Strategies and Model Serving using FastAPI

Use the Kaggle Titanic labelled `train.csv`: https://www.kaggle.com/competitions/titanic/data

Run all cells from top to bottom. The model is trained once, saved as `model.pkl`, and reused for all inference methods and FastAPI serving.

In [1]:
!pip -q install pandas numpy scikit-learn joblib fastapi uvicorn pydantic requests httpx

## Upload the dataset
Download `train.csv` from Kaggle, then upload it below.

In [2]:
from google.colab import files
uploaded = files.upload()

Saving titanic.csv to titanic.csv


In [3]:
import os, time, joblib, pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

DATA_PATH = "train.csv"
if not os.path.exists(DATA_PATH):
    choices = [x for x in os.listdir(".") if x.lower().endswith(".csv")]
    if len(choices) == 1: DATA_PATH = choices[0]
    else: raise FileNotFoundError("Upload Kaggle Titanic train.csv first.")
df = pd.read_csv(DATA_PATH)
display(df.head())
print("Dataset shape:", df.shape)

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


Dataset shape: (891, 15)


## Task 1 — Preprocess, train, evaluate, and save the pipeline

In [7]:
# Map actual columns (ignoring case) to the exact names expected by the model pipeline
column_mapping = {
    "survived": "Survived",
    "pclass": "Pclass",
    "sex": "Sex",
    "age": "Age",
    "sibsp": "SibSp",
    "parch": "Parch",
    "fare": "Fare",
    "embarked": "Embarked"
}
df = df.rename(columns={c: column_mapping[c.lower()] for c in df.columns if c.lower() in column_mapping})

NUMERIC = ["Pclass", "Age", "SibSp", "Parch", "Fare"]
CATEGORICAL = ["Sex", "Embarked"]
FEATURES = NUMERIC + CATEGORICAL
TARGET = "Survived"
missing = [c for c in FEATURES + [TARGET] if c not in df.columns]
if missing: raise ValueError(f"Missing required columns: {missing}")
df = df.dropna(subset=[TARGET]).copy()
X, y = df[FEATURES], df[TARGET].astype(int)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
preprocessor = ColumnTransformer([
    ("numeric", Pipeline([("imputer", SimpleImputer(strategy="median")),
                          ("scaler", StandardScaler())]), NUMERIC),
    ("categorical", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                              ("onehot", OneHotEncoder(handle_unknown="ignore"))]), CATEGORICAL)
])
model = Pipeline([("preprocessing", preprocessor),
                  ("classifier", RandomForestClassifier(n_estimators=200, random_state=42,
                                                        class_weight="balanced"))])
model.fit(X_train, y_train)
test_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, test_pred)
joblib.dump(model, "model.pkl")
heldout = X_test.copy()
heldout["Survived"] = y_test.values
heldout.to_csv("heldout_passengers.csv", index=False)
print(f"Test accuracy: {accuracy:.4f} ({accuracy:.2%})")
print("Saved model.pkl and heldout_passengers.csv")
print(classification_report(y_test, test_pred, zero_division=0))

Test accuracy: 0.8156 (81.56%)
Saved model.pkl and heldout_passengers.csv
              precision    recall  f1-score   support

           0       0.82      0.89      0.86       110
           1       0.80      0.70      0.74        69

    accuracy                           0.82       179
   macro avg       0.81      0.79      0.80       179
weighted avg       0.81      0.82      0.81       179



## Task 2 — Online inference (one unseen passenger)

In [8]:
loaded_model = joblib.load("model.pkl")
one = heldout[FEATURES].iloc[[0]]
one_pred = int(loaded_model.predict(one)[0])
display(one)
print("Prediction:", one_pred, "|", "Survived" if one_pred else "Did not survive")
print("Actual held-out label:", int(heldout["Survived"].iloc[0]))

,Pclass,Age,SibSp,Parch,Fare,Sex,Embarked
565,3,24.0,2,0,24.15,male,S


Prediction: 0 | Did not survive
Actual held-out label: 0


## Task 3 — Batch inference (20 unseen passengers)

In [9]:
if len(heldout) < 20: raise ValueError("Need at least 20 held-out passengers.")
batch = heldout[FEATURES].iloc[:20].copy()
batch_result = batch.copy()
batch_result["Predicted_Survived"] = loaded_model.predict(batch).astype(int)
batch_result["Actual_Survived"] = heldout["Survived"].iloc[:20].astype(int).values
batch_result.to_csv("batch_predictions.csv", index=False)
display(batch_result)
print("Saved batch_predictions.csv")

,Pclass,Age,SibSp,Parch,Fare,Sex,Embarked,Predicted_Survived,Actual_Survived
565,3,24.0,2,0,24.1500,male,S,0,0
160,3,44.0,0,1,16.1000,male,S,0,0
553,3,22.0,0,0,7.2250,male,C,0,1
860,3,41.0,2,0,14.1083,male,S,0,0
241,3,NaN,1,0,15.5000,female,Q,1,1
559,3,36.0,1,0,17.4000,female,S,0,1
387,2,36.0,0,0,13.0000,female,S,1,1
536,1,45.0,0,0,26.5500,male,S,1,0
698,1,49.0,1,1,110.8833,male,C,0,0
99,2,34.0,1,0,26.0000,male,S,0,0


Saved batch_predictions.csv


## Task 4 — Streaming inference (10 records, one-second delay)

In [10]:
stream_predictions = []
for i in range(10):
    row = heldout[FEATURES].iloc[[i]]
    pred = int(loaded_model.predict(row)[0])
    stream_predictions.append(pred)
    print(f"Incoming record {i+1}/10: predicted Survived={pred}")
    if i < 9: time.sleep(1)

Incoming record 1/10: predicted Survived=0
Incoming record 2/10: predicted Survived=0
Incoming record 3/10: predicted Survived=0
Incoming record 4/10: predicted Survived=0
Incoming record 5/10: predicted Survived=1
Incoming record 6/10: predicted Survived=0
Incoming record 7/10: predicted Survived=1
Incoming record 8/10: predicted Survived=1
Incoming record 9/10: predicted Survived=0
Incoming record 10/10: predicted Survived=0


## Task 5 — FastAPI endpoints and valid/invalid request tests

In [11]:
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field, ConfigDict
from fastapi.testclient import TestClient

api_model = joblib.load("model.pkl")
app = FastAPI(title="Titanic Survival Prediction API", version="1.0.0")

class Passenger(BaseModel):
    model_config = ConfigDict(extra="forbid")
    Pclass: int = Field(ge=1, le=3)
    Sex: str
    Age: float | None = Field(default=None, ge=0, le=120)
    SibSp: int = Field(ge=0, le=20)
    Parch: int = Field(ge=0, le=20)
    Fare: float = Field(ge=0)
    Embarked: str | None = "S"

@app.get("/health")
def health():
    return {"status": "ok", "model_loaded": api_model is not None}

@app.post("/predict")
def predict(passenger: Passenger):
    row = pd.DataFrame([passenger.model_dump()])[FEATURES]
    pred = int(api_model.predict(row)[0])
    return {"prediction": pred, "label": "Survived" if pred else "Did not survive"}

api_client = TestClient(app)
health_response = api_client.get("/health")
print("GET /health:", health_response.status_code, health_response.json())
valid = {"Pclass":3,"Sex":"female","Age":22,"SibSp":1,"Parch":0,"Fare":7.25,"Embarked":"S"}
valid_response = api_client.post("/predict", json=valid)
print("Valid POST /predict:", valid_response.status_code, valid_response.json())
invalid = {"Pclass":9,"Sex":"female","Age":22,"SibSp":1,"Parch":0,"Fare":7.25,"Embarked":"S"}
invalid_response = api_client.post("/predict", json=invalid)
print("Invalid POST /predict:", invalid_response.status_code, invalid_response.json())
assert health_response.status_code == 200
assert valid_response.status_code == 200
assert invalid_response.status_code == 422
print("All API endpoint tests passed.")

GET /health: 200 {'status': 'ok', 'model_loaded': True}
Valid POST /predict: 200 {'prediction': 1, 'label': 'Survived'}
Invalid POST /predict: 422 {'detail': [{'type': 'less_than_equal', 'loc': ['body', 'Pclass'], 'msg': 'Input should be less than or equal to 3', 'input': 9, 'ctx': {'le': 3}}]}
All API endpoint tests passed.


### Optional: launch the live API server

In [12]:
# To use Swagger UI, save the app to app.py (provided in the project ZIP), then run:
# !uvicorn app:app --host 0.0.0.0 --port 8000
# In Colab, a local server is not automatically public; a tunnel is needed for browser access.

## Task 6 — Observations

1. **Online vs Batch:** Online predicts one record on demand; batch predicts many records together.
2. **Streaming vs Batch:** Streaming handles records one at a time as they arrive; batch processes a fixed collection together.
3. **Why FastAPI:** It exposes predictions via HTTP, validates data with Pydantic, and provides interactive Swagger documentation at `/docs`.
4. **Why reuse the model:** The same saved pipeline ensures consistent preprocessing and predictions across all inference methods.

In [13]:
from google.colab import files
import zipfile
with zipfile.ZipFile("CLA4_generated_outputs.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for filename in ["model.pkl", "heldout_passengers.csv", "batch_predictions.csv"]:
        if os.path.exists(filename): z.write(filename)
files.download("CLA4_generated_outputs.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>